In [1]:
import layers
import modules
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from scipy.signal import resample_poly
from scipy.signal import butter, sosfiltfilt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import os
import pandas as pd
from tensorflow.keras.callbacks import ModelCheckpoint

from AF_metrics_utils import *

In [2]:
train_idx = np.load('/home/tsu25/ECG_PWD/Fetal-maternal-fusion/src/idx_train_risk.npy')
val_idx = np.load('/home/tsu25/ECG_PWD/Fetal-maternal-fusion/src/idx_val_risk.npy')

Y = np.load('/home/tsu25/ECG_PWD/Fetal-maternal-fusion/src/WaveNet_beat/data/Y.npy')
X = np.load('/home/tsu25/ECG_PWD/Fetal-maternal-fusion/src/WaveNet_beat/data/X.npy')

In [7]:
save_dir_model = './WaveNet_beat/low_risk_models'
save_dir_loss  = './WaveNet_beat/low_risk_logs'
save_dir_plots = './WaveNet_beat/low_risk_plots'
save_dir_generated = './WaveNet_beat/generated_data' 

os.makedirs(save_dir_model, exist_ok=True)
os.makedirs(save_dir_loss,  exist_ok=True)
os.makedirs(save_dir_plots, exist_ok=True)
os.makedirs(save_dir_generated, exist_ok=True)

latent_dim = 1065

best_model = modules.WaveNet_two_channel(input_shape=(latent_dim, 2))
best_model.compile(optimizer="adam", loss="mae")
best_model.load_weights(os.path.join(save_dir_model, f'best_single_model.weights.h5'))

model = modules.WaveNet_two_channel(input_shape=(latent_dim, 2))
model.compile(optimizer="adam", loss="mae")
model.load_weights(os.path.join(save_dir_model, f'final_model.weights.h5'))

selected_ecgs = X[val_idx]
DUS_array_test = Y[val_idx]

best_model_generated_dopplers = best_model.predict(selected_ecgs, verbose=0).squeeze(-1)
final_model_generated_dopplers = model.predict(selected_ecgs, verbose=0).squeeze(-1)

Model: "WaveNet_two_channel_early_fusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ecg_fetal_maternal… │ (None, 1065, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 1065, 64)  │        192 │ ecg_fetal_matern… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 1065, 64)  │     81,984 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 1065, 64)  │     81,984 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1065, 64)  │          0 │ conv1d_1[0][0],   │
│                     │                   │            │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 1065, 64)  │      4,160 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1065, 64)  │          0 │ multiply[0][0],   │
│                     │                   │            │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 1065, 64)  │     81,984 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 1065, 64)  │     81,984 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 1065, 64)  │          0 │ conv1d_5[0][0],   │
│ (Multiply)          │                   │            │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 1065, 64)  │      4,160 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 1065, 64)  │          0 │ multiply_1[0][0], │
│                     │                   │            │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 1065, 64)  │     81,984 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 1065, 64)  │          0 │ conv1d_9[0][0],   │
│ (Multiply)          │                   │            │ conv1d_10[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 1065, 64)  │      4,160 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 1065, 64)  │          0 │ multiply_2[0][0], │
│                     │                   │            │ conv1d_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 1065, 64)  │          0 │ conv1d_13[0][0],  │
│ (Multiply)          │                   │            │ conv1d_14[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 1065, 64)  │      4,160 │ add_2[0][0]     

 Total params: 1,206,273 (4.60 MB)

 Trainable params: 1,206,273 (4.60 MB)

 Non-trainable params: 0 (0.00 B)

...

Model: "WaveNet_two_channel_early_fusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ecg_fetal_maternal… │ (None, 1065, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_31 (Conv1D)  │ (None, 1065, 64)  │        192 │ ecg_fetal_matern… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_32 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ conv1d_31[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_33 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ conv1d_31[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_7          │ (None, 1065, 64)  │          0 │ conv1d_32[0][0],  │
│ (Multiply)          │                   │            │ conv1d_33[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_35 (Conv1D)  │ (None, 1065, 64)  │      4,160 │ conv1d_31[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 1065, 64)  │          0 │ multiply_7[0][0], │
│                     │                   │            │ conv1d_35[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_36 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_8[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_37 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_8[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_8          │ (None, 1065, 64)  │          0 │ conv1d_36[0][0],  │
│ (Multiply)          │                   │            │ conv1d_37[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_39 (Conv1D)  │ (None, 1065, 64)  │      4,160 │ add_8[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 1065, 64)  │          0 │ multiply_8[0][0], │
│                     │                   │            │ conv1d_39[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_40 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_41 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_9          │ (None, 1065, 64)  │          0 │ conv1d_40[0][0],  │
│ (Multiply)          │                   │            │ conv1d_41[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_43 (Conv1D)  │ (None, 1065, 64)  │      4,160 │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 1065, 64)  │          0 │ multiply_9[0][0], │
│                     │                   │            │ conv1d_43[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_44 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_10[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_45 (Conv1D)  │ (None, 1065, 64)  │     81,984 │ add_10[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_10         │ (None, 1065, 64)  │          0 │ conv1d_44[0][0],  │
│ (Multiply)          │                   │            │ conv1d_45[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_47 (Conv1D)  │ (None, 1065, 64)  │      4,160 │ add_10[0][0]    

 Total params: 1,206,273 (4.60 MB)

 Trainable params: 1,206,273 (4.60 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
save_dir_generated = './WaveNet_beat/generated_data' 
os.makedirs(save_dir_generated, exist_ok=True)

np.save(os.path.join(save_dir_generated, 'low_risk_fusion_model_final.npy'), 
        final_model_generated_dopplers)
np.save(os.path.join(save_dir_generated, 'low_risk_fusion_model_best.npy'), 
        best_model_generated_dopplers)